In [ ]:
import pyautogui          #more library installation
import pytesseract
import cv2
import numpy as np
import pandas as pd
import time
import math
import sys
import re
import statistics
import os
import subprocess
import requests
import keyboard


In [ ]:
#Installs pytesseract for scanning
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


In [ ]:
pyautogui.FAILSAFE = True;   #security mechanism 

def locate_coordinates():     #function to find coordinates on computer screen
    print("Move your mouse to see coordinates. Press Ctrl+C to stop.\n")
    try:
        while True:
            # Get current mouse position
            x, y = pyautogui.position()
            # Print coordinates in a fixed-width format
            print(f"X: {x:4}  Y: {y:4}", end="\r", flush=True)
            time.sleep(0.05)  # Update every 50ms
    except KeyboardInterrupt:
        print("\nStopped tracking.")
        sys.exit(0)
    except Exception as e:
        print(f"\nError: {e}")
        sys.exit(1)

if __name__ == "__main__":
    locate_coordinates()

In [ ]:
# Load coordinates of buttons
xcoor = dict(activate_rocker = 81, left_rocker = 540, right_rocker = 1440, unlock_lift = 73 , lift_off = 74, take_picture = 77, toggle_speed=78, lift_slide1 = 777, lift_slide2 = 1148);
ycoor = dict(activate_rocker = 522, left_rocker = 579, right_rocker = 576 , unlock_lift = 985 , lift_off = 826, take_picture = 239, toggle_speed=674, lift_slide1 = 592, lift_slide2 = 585);
xtemp = dict(r1_up = 536, r1_down = 538, r1_left = 362, r1_right = 713);
ytemp = dict(r1_up = 386, r1_down = 763, r1_left = 576, r1_right = 565);
xcoor.update(xtemp);
ycoor.update(ytemp);
xtemp = dict(r2_up = 1438, r2_down = 1435, r2_left = 1251, r2_right = 1620, cam_tilt_mid = 202, cam_tilt_down = 202 );
ytemp = dict(r2_up = 384, r2_down = 760, r2_left = 572, r2_right = 569, cam_tilt_mid = 836, cam_tilt_down = 1003);
xcoor.update(xtemp);
ycoor.update(ytemp);

In [ ]:
# Initiate flags
rockers_activated = False;
in_flight = False;
exit_program = False;

In [ ]:
# Define commands
def activate_rockers():
    global rockers_activated;
    if(not rockers_activated):
        pyautogui.click(xcoor['activate_rocker'], ycoor['activate_rocker']);
        rockers_activated = True;
        
def deactivate_rockers():
    global rockers_activated;
    if(rockers_activated):
        pyautogui.click(xcoor['activate_rocker'], ycoor['activate_rocker']);
        rockers_activated = False;
        
def move_drone(direction,duration):
    latency = 0.25;
    pyautogui.mouseUp();
    if(not rockers_activated):
        activate_rockers();
    if(direction == 'forward'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_up'], ycoor['r2_up'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_up'], ycoor['r2_up'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'backward'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_down'], ycoor['r2_down'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_down'], ycoor['r2_down'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'left'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_left'], ycoor['r2_left'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_left'], ycoor['r2_left'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'right'):
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
        pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
        pyautogui.moveTo(xcoor['r2_right'], ycoor['r2_right'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r2_right'], ycoor['r2_right'], button='left')
        pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
    elif(direction == 'up'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_up'], ycoor['r2_up'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_up'], ycoor['r1_up'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
    elif(direction == 'down'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_down'], ycoor['r1_down'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_down'], ycoor['r1_down'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        
def drive_drone(direction,speed=0.7):
    global cur_direction;
    global cur_speed;
    latency = 0.25;
    #Check for change in direction
    if(cur_direction != direction):
        pyautogui.mouseUp();
        cur_direction = direction;
        #Reposition mouse to appropriate rocker
        if(direction == "up" or direction == "down"):
            pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
            pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        else:
            pyautogui.moveTo(xcoor['right_rocker'], ycoor['right_rocker']);
            pyautogui.mouseDown(xcoor['right_rocker'], ycoor['right_rocker'], button='left')
    elif(cur_speed == speed):
        return
        #Only rejusting speed
   # else:
        #Direction and speed unchanged, ignore call
                
        
    #Restrict speed from 0 to 1
    if(speed>1):
        speed = 1;
    if(speed<0):
        speed = 0;
    cur_speed = speed;
    #Define rocker movement vector and scale it with speed
    tiltx = (xcoor['r2_right'] - xcoor['right_rocker'])*speed;
    tilty = (ycoor['r2_up'] - ycoor['right_rocker'])*speed;
    tiltz = (ycoor['r1_up'] - ycoor['left_rocker'])*speed;
    #Activate rockers if needed
    if(not rockers_activated):
        activate_rockers();
    #Tilt rockers in the designated direction with the appropriate speed
    if(direction == 'forward'):
        pyautogui.moveTo(xcoor['r2_up'], ycoor['right_rocker'] + tilty , duration=latency);
    elif(direction == 'backward'):
        pyautogui.moveTo(xcoor['r2_down'], ycoor['right_rocker'] - tilty , duration=latency);
    elif(direction == 'right'):
        pyautogui.moveTo(xcoor['right_rocker'] + tiltx, ycoor['r2_right'], duration=latency);
    elif(direction == 'left'):
        pyautogui.moveTo(xcoor['right_rocker'] - tiltx, ycoor['r2_left'], duration=latency);
    elif(direction == 'up'):
        pyautogui.moveTo(xcoor['r1_up'], ycoor['left_rocker'] + tiltz, duration=latency);
    elif(direction == 'down'):
        pyautogui.moveTo(xcoor['r1_down'], ycoor['left_rocker'] - tiltz, duration=latency);
        
def turn_drone(direction,duration):
    latency = 0.25;
    pyautogui.mouseUp();
    if(not rockers_activated):
        activate_rockers();
    if(direction == 'left'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_left'], ycoor['r1_left'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_left'], ycoor['r1_left'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
    elif(direction == 'right'):
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        pyautogui.mouseDown(xcoor['left_rocker'], ycoor['left_rocker'], button='left')
        pyautogui.moveTo(xcoor['r1_right'], ycoor['r1_right'], duration=latency);
        time.sleep(duration)
        pyautogui.mouseUp(xcoor['r1_right'], ycoor['r1_right'], button='left')
        pyautogui.moveTo(xcoor['left_rocker'], ycoor['left_rocker']);
        
def unlock_press(delay=0.5):
    time.sleep(delay)
    pyautogui.click(xcoor['unlock_lift'], ycoor['unlock_lift'], duration=1);
    time.sleep(delay)
    
def lift_slide(delay=0.5):
    global in_flight;
    if(not in_flight):
        time.sleep(delay)
        pyautogui.click(xcoor['lift_off'], ycoor['lift_off'])
        pyautogui.mouseDown(xcoor['lift_slide1'], ycoor['lift_slide1'], button='left')
        pyautogui.moveTo(xcoor['lift_slide2'], ycoor['lift_slide2'], duration=1);
        pyautogui.mouseUp(xcoor['lift_slide2'], ycoor['lift_slide2'], button='left')
        time.sleep(delay)
        in_flight = True;
     

    
def camera_tilt(direc, durat):
    time.sleep(0.5)
    pyautogui.mouseDown(xcoor['cam_tilt_mid'], ycoor['cam_tilt_mid'], button='left')
    if(direc == 'up'):
        pyautogui.moveTo(xcoor['cam_tilt_up'], ycoor['cam_tilt_up'], duration=1)
    elif(direc == 'down'):
        pyautogui.moveTo(xcoor['cam_tilt_down'], ycoor['cam_tilt_down'], duration=1)
    time.sleep(durat)
    pyautogui.mouseUp(xcoor['cam_tilt_mid'], ycoor['cam_tilt_mid'], button='left')
    time.sleep(0.5)

In [ ]:
def stop_drone():
    pyautogui.mouseUp(button='left')

In [ ]:
def land_drone():
    time.sleep(1)
    pyautogui.click(xcoor['lift_off'], ycoor['lift_off'])
    pyautogui.mouseDown(xcoor['lift_slide1'], ycoor['lift_slide1'], button='left')
    pyautogui.moveTo(xcoor['lift_slide2'], ycoor['lift_slide2'], duration=1)
    pyautogui.mouseUp(xcoor['lift_slide2'], ycoor['lift_slide2'], button='left')
    time.sleep(1)

In [ ]:
# Create key press exit
def quit_program():
    global exit_program
    print("\nHotkey pressed. Exiting...")
    exit_program = True

keyboard.add_hotkey('q', quit_program)

In [ ]:

def clean_coord_num(s: str) -> str:    #function receives a string and outputs a string 
    s = s.replace(',', '.')     #replace comma with period
    s = re.sub(r'[^0-9\.\-]', '', s)  #redefines inout string to replace anything not a neg sign, number, or dec point with space
    return s

# Function to capture a specific region using bounding box coordinates (left, top, width, height)

def read_coords():
    region = (920, 35, 150, 80)   
    region_screenshot = pyautogui.screenshot(region=region)
    img = cv2.cvtColor(np.array(region_screenshot), cv2.COLOR_BGR2GRAY)

    _, processed_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    text = pytesseract.image_to_string(processed_img)
    text = text.replace('o','0').replace('O','0')
    lines = text.splitlines()
    
    #print(repr(text))
    #print(lines)

    try:   # this prevents error values from being accepted
        latitude_read = float(clean_coord_num(lines[1]))
        longitude_read = float(clean_coord_num(lines[0]))
        
        if 'W' in lines[0] and longitude_read>0:
            longitude_read = -longitude_read

        return latitude_read, longitude_read

    except (IndexError, ValueError):
        return None, None
    
    #averages the coordiantes found for better control

def read_coords_stable(n=3, delay=0.04):
    lats, lons = [], []

    for _ in range(n):
        lat, lon = read_coords()
        if lat is not None and lon is not None:
            lats.append(lat)
            lons.append(lon)
        time.sleep(delay)

    if not lats:
        return None, None

    return statistics.median(lats), statistics.median(lons)


In [ ]:
def clean_alt_num(s: str) -> str:    #function receives a string and outputs a string 
    s = s.replace(',', '.')     #replace comma with period
    s = s.replace('m', '')   #replace m with space
    s = re.sub(r'[^0-9\.\-]', '', s)  #redefines inout string to replace anything not a neg sign, number, or dec point with space
    return s
#read altitude

def read_altitude():
    alt_region = (500, 30, 65, 36)   
    alt_region_screenshot = pyautogui.screenshot(region=alt_region)
    img = cv2.cvtColor(np.array(alt_region_screenshot), cv2.COLOR_BGR2GRAY)

    _, processed_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    text = pytesseract.image_to_string(processed_img)
    text = text.replace('o','0').replace('O','0')
    
    

    try:   # this prevents error values from being accepted
        alt_read = float(clean_alt_num(text))
        
        if alt_read < 0 or alt_read > 60:
            return None
        
        return alt_read

    except (IndexError, ValueError):
        return None
#averages the coordiantes found for better control

def read_alt_stable(n=3, delay=0.04):
    alts = []

    for _ in range(n):
        alt = read_altitude()
        if alt is not None:
            alts.append(alt)
        time.sleep(delay)

    if not alts:
        return None

    return statistics.median(alts)


In [ ]:
def clean_batt_num(s: str) -> str:    #function receives a string and outputs a string 
    s = s.replace('%', '')   #replace % with space
    s = re.sub(r'[^0-9]', '', s)  #redefines inout string to replace anything not number with space
    return s
#read battery

def read_batt():
    batt_region = (295, 93, 65, 30)   
    batt_region_screenshot = pyautogui.screenshot(region=batt_region)
    img = cv2.cvtColor(np.array(batt_region_screenshot), cv2.COLOR_BGR2GRAY)

    _, processed_img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)

    text = pytesseract.image_to_string(processed_img)
    text = text.replace('o','0').replace('O','0')
    
    

    try:   # this prevents error values from being accepted
        batt_read = int(clean_batt_num(text))
        
        if batt_read == "":
            return None
        
        if batt_read < 0 or batt_read > 100:
            return None
        
        return batt_read

    except (IndexError, ValueError):
        return None
#averages the coordiantes found for better control

def read_batt_stable(n=3, delay=0.04):
    batts = []

    for _ in range(n):
        batt = read_batt()
        if batt is not None:
            batts.append(batt)
        time.sleep(delay)

    if not batts:
        return None

    return statistics.median(batts)


In [ ]:
BASE_URL = "https://parking.2759359719sw.workers.dev"

In [ ]:
def take_picture(xcoor, ycoor): #basically makes a pipeline: connect to adb, press cam, wait for new file, put in local folder

    adb_path = r"C:\Platform_Tools\platform-tools\adb.exe"    
    adb_ip = "127.0.0.1:5555"                             
    blue_img_folder = "/sdcard/DCIM/LWProPhotos"            
    local = "Drone_Pics"                             

    # connect adb
    subprocess.run([adb_path, "connect", adb_ip], capture_output=True, text=True) # shell: adb.exe connect adb_ip

    # finds a list of the pics before new img capture so that old ones not reused 
    result = subprocess.run(
        [adb_path, "-s", adb_ip, "shell", "ls", blue_img_folder], #shell:C:adb_path -s adb_ip shell ls blue_img_folder
        capture_output=True,
        text=True
    )  #this will spit img file numbers horizontally

    files_before = result.stdout.splitlines()    #puts the img file numbers vertically

    #press app camera
    x = xcoor["take_picture"]
    y = ycoor["take_picture"]
    pyautogui.click(x, y)

    time.sleep(2)

    start_time = time.time()
    new_file = None

    # checks for new files 
    while time.time() - start_time < 10:  #gives a ten second period for new img upload (reduce for faster response later)

        result = subprocess.run(
            [adb_path, "-s", adb_ip, "shell", "ls", blue_img_folder],
            capture_output=True,
            text=True
        )

        files_after = result.stdout.splitlines()  # spits all img names vertically agian

        for i in files_after:   #checks each line with img name 
            if i not in files_before:
                new_file = i   # if string doesnt match, it assumes new img file
                break

        if new_file:  # leave while loop when new file exists
            break

        time.sleep(0.5)

    if new_file is None:
        return None

    # puts where new file would be redirected to
    new_img_path = os.path.join(local, new_file)

    subprocess.run([    #actually copies new img file to local folder 
        adb_path,
        "-s",
        adb_ip,
        "pull",
        blue_img_folder + "/" + new_file,
        new_img_path
    ], capture_output=True, text=True)

    return new_img_path

In [ ]:

def get_lots():
    url = f"{BASE_URL}/api/lot"
    response = requests.get(url)
    return response.json()

def get_spaces():
    url = f"{BASE_URL}/api/space"
    response = requests.get(url)
    return response.json()

In [ ]:
# function used to upload pictures
def upload_frame(new_img_path, BASE_URL):
  with open(new_img_path, 'rb') as f:
    response = requests.post(f"{BASE_URL}/api/upload-frame", data=f.read(), params={'filename': new_img_path.split('/')[-1]})
    return response.json()


In [ ]:
#main drone control function 
#checkpoints = [
#    (26.4382271, -80.1050785),  ##first checkpoint   this has an index of 0 for reference 
#    (26.4638297, -80.2178668),   ##second checkpoint  this has an index of 1 for reference, we can add more for the future too 
#    (26.4382271, -80.1050785)   #third checkpoint goes back to first
#]

checkpoints = [
    (26.4638241, -80.2177077),  ##park coordinates
    (26.4638297, -80.2178668),   
    (26.4638218, -80.217769)
]

#checkpoints = [
 #   (26.463795, -80.217748), 
  #  (26.4637543, -80.2168824),   ##park2 coordinates 
   # (26.463795, -80.217748)
 #]
    
    
current_cp = 0   ##start at the first checkpoint 

lat_thresh = 0.00002   ## based on some measurements done on google earth
lon_thresh = 0.00002    ## based on some measurements done on google earth

max_altitude = 55  #based on prior imaging 


In [ ]:
time.sleep(5)
unlock_press()
lift_slide()
activate_rockers()


camera_tilt('down', 4)

while not exit_program:  
    altitude_read = read_alt_stable(n=3, delay=0.08)

    if altitude_read is None:
        time.sleep(0.1)
        continue

    alt_error = max_altitude - altitude_read

    if abs(alt_error) <= 5:
        stop_drone()
        print("Drone at Height")
        break

    if alt_error > 0:
        move_drone('up', 1.2)
    else:
        move_drone('down', 1.2)

    stop_drone()
    
    time.sleep(2)


low_batt_count = 0
loop_counter = 0

while not exit_program:
    latitude_read, longitude_read = read_coords_stable(n=3, delay=0.04)

    loop_counter += 1
    if loop_counter % 5 == 0:
        battery = read_batt_stable(n=3, delay=0.04)

        if battery is not None and battery < 30:
            low_batt_count += 1
        else:
            low_batt_count = 0

        if low_batt_count >= 3:
            print("LOW POWER!!")
            land_drone()
            break
        
    if latitude_read is None or longitude_read is None:
        time.sleep(0.1)
        continue

    cp_lat, cp_lon = checkpoints[current_cp]  ##the checkpoint lats and longs are upodated based on the list 


    if abs(latitude_read - cp_lat) <= lat_thresh and abs(longitude_read - cp_lon) <= lon_thresh:  # for whichever checkpoint since the logic is the same, if the thresholds are met go to the next one
        time.sleep(2)
        file_path = take_picture(xcoor, ycoor)
        
        if file_path:   #if statement to check if upload happened so no crash
            upload_frame(file_path, BASE_URL) 
            print("Image Uploaded!!")
        current_cp += 1  ##goes to next index or checkpoint on list for next coordinates
        time.sleep(1)

        if current_cp >= len(checkpoints):  ## once the number of elemetns are exceeded on the checkpoint list, land the drone and break
            # Land
            land_drone()
            break

        time.sleep(1)   ##small delayq
        continue

    if latitude_read > cp_lat + lat_thresh:     ## the code always adjusts the position from IP to CP1 or CP1 to CP2
        #move drone south
        move_drone('backward',0.5);

    elif latitude_read < cp_lat - lat_thresh:
        #move drone north
        move_drone('forward',0.5);

    if longitude_read > cp_lon + lon_thresh:
        #move drone west
        move_drone('left',0.5);

    elif longitude_read < cp_lon - lon_thresh:
        #move drone east
        move_drone('right',0.5);
        
    time.sleep(0.2)
        

